In [1]:
from pathlib import Path
import sys

# common/ lives in BuildingWithClaudeAPI/, a sibling of this folder.
# Walk up from the current directory until BuildingWithClaudeAPI/common is found.
sys.path.append(r"C:\Users\SValathalaKu\Data\claude-bedrock-learning\BuildingWithClaudeAPI")
from common.aws_config import aws_env_setup
#from common.bedrock_config import get_anthropic_client

aws_env_setup()
#client, model = get_anthropic_client()


In [2]:
from anthropic import AnthropicBedrock

def get_anthropic_client():

     client = AnthropicBedrock(
      aws_region="ap-southeast-2"
      )
     model = "au.anthropic.claude-haiku-4-5-20251001-v1:0"
     return client, model
client, model = get_anthropic_client()

In [ ]:
import asyncio
import json
from pathlib import Path
from anthropic import AnthropicBedrock


# ============================================================
# 1. CLAUDE CLIENT
# ============================================================

client = AnthropicBedrock(
    aws_region="ap-southeast-2"
)

model = "au.anthropic.claude-haiku-4-5-20251001-v1:0"


# ============================================================
# 2. REAL TOOL FUNCTIONS
#
# These functions perform deterministic work.
# Claude does not read the files directly.
# ============================================================

def get_claim(claim_id: str) -> dict:
    """Retrieve one claim from claims.json."""

    claims = json.loads(
        Path("claims.json").read_text(encoding="utf-8")
    )

    if claim_id not in claims:
        raise ValueError(f"Claim not found: {claim_id}")

    return claims[claim_id]


def get_policy(policy_id: str) -> dict:
    """Retrieve one policy from policies.json."""

    policies = json.loads(
        Path("policies.json").read_text(encoding="utf-8")
    )

    if policy_id not in policies:
        raise ValueError(f"Policy not found: {policy_id}")

    return policies[policy_id]


# ============================================================
# 3. TOOL SCHEMAS
#
# Each subagent receives only the tool needed for its role.
# This is tool restriction / least privilege.
# ============================================================

claim_tool = {
    "name": "get_claim",
    "description": (
        "Retrieve the current status, rejection reason, service details "
        "and claimed amount for one insurance claim. "
        "Use for claim-processing facts. "
        "Do not use for policy coverage rules."
    ),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "claim_id": {
                "type": "string",
                "description": "Claim identifier, for example CLM-1048."
            }
        },
        "required": ["claim_id"],
        "additionalProperties": False
    }
}


policy_tool = {
    "name": "get_policy",
    "description": (
        "Retrieve coverage rules, benefit percentages, limits and "
        "authorisation requirements for one insurance policy. "
        "Use for policy interpretation. "
        "Do not use for the current status of a claim."
    ),
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "policy_id": {
                "type": "string",
                "description": (
                    "Policy identifier, for example GOLD-HOSPITAL."
                )
            }
        },
        "required": ["policy_id"],
        "additionalProperties": False
    }
}


# ============================================================
# 4. GENERIC SUBAGENT
#
# This is an agent because it has:
# - a goal
# - its own Claude conversation
# - restricted tools
# - an agentic loop
# - freedom to decide when to use its tool
# ============================================================

def run_subagent(
    subagent_name: str,
    goal: str,
    context: dict,
    tools: list,
    tool_functions: dict,
    max_turns: int = 4
) -> dict:

    messages = [
        {
            "role": "user",
            "content": f"""
You are the {subagent_name}.

Goal:
{goal}

Context explicitly supplied by the coordinator:
{json.dumps(context, indent=2)}

Use the available tool to investigate.

Return:
1. Confirmed findings
2. Supporting evidence
3. Missing or uncertain information
4. Recommended next action

Do not invent facts.
"""
        }
    ]

    # Agentic loop
    for turn in range(max_turns):

        response = client.messages.create(
            model=model,
            max_tokens=500,
            tools=tools,
            tool_choice={"type": "auto"},
            messages=messages
        )

        # end_turn means the subagent has finished.
        if response.stop_reason == "end_turn":
            text = "\n".join(
                block.text
                for block in response.content
                if block.type == "text"
            )

            return {
                "subagent": subagent_name,
                "status": "completed",
                "finding": text
            }

        # tool_use means the subagent needs external information.
        if response.stop_reason != "tool_use":
            return {
                "subagent": subagent_name,
                "status": "failed",
                "error": (
                    f"Unexpected stop reason: "
                    f"{response.stop_reason}"
                )
            }

        # Preserve Claude's complete tool request.
        messages.append({
            "role": "assistant",
            "content": response.content
        })

        tool_results = []

        for block in response.content:
            if block.type != "tool_use":
                continue

            try:
                function = tool_functions.get(block.name)

                if function is None:
                    raise ValueError(
                        f"Unknown tool: {block.name}"
                    )

                # Convert Claude's input dictionary into function arguments.
                output = function(**block.input)

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output)
                })

            except ValueError as error:
                # Validation failure: retrying unchanged will not help.
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps({
                        "errorCategory": "validation",
                        "isRetryable": False,
                        "description": str(error)
                    }),
                    "is_error": True
                })

            except OSError as error:
                # Transient failure: a retry might succeed.
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps({
                        "errorCategory": "transient",
                        "isRetryable": True,
                        "description": str(error)
                    }),
                    "is_error": True
                })

        # Return every result in one user message.
        messages.append({
            "role": "user",
            "content": tool_results
        })

    return {
        "subagent": subagent_name,
        "status": "failed",
        "error": "Maximum subagent turn limit reached."
    }


# ============================================================
# 5. COORDINATOR
#
# The coordinator:
# - creates explicit shared context
# - delegates specialised goals
# - runs independent subagents concurrently
# - synthesises their findings
# ============================================================

async def coordinator(
    claim_id: str,
    policy_id: str,
    user_question: str
) -> str:

    # Subagents do NOT automatically inherit coordinator context.
    shared_context = {
        "claim_id": claim_id,
        "policy_id": policy_id,
        "user_question": user_question
    }

    # Claims agent receives only get_claim.
    claims_job = asyncio.to_thread(
        run_subagent,
        "Claims Investigation Subagent",
        "Determine why the claim was rejected.",
        shared_context,
        [claim_tool],
        {"get_claim": get_claim}
    )

    # Policy agent receives only get_policy.
    policy_job = asyncio.to_thread(
        run_subagent,
        "Policy Coverage Subagent",
        (
            "Determine whether the claimed service is covered "
            "and identify the applicable benefit rules."
        ),
        shared_context,
        [policy_tool],
        {"get_policy": get_policy}
    )

    # Independent investigations run concurrently.
    claims_result, policy_result = await asyncio.gather(
        claims_job,
        policy_job
    )

    # Structured handoff to the coordinator.
    synthesis_prompt = f"""
You are the coordinator.

Original question:
{user_question}

Shared identifiers:
{json.dumps(shared_context, indent=2)}

Claims subagent result:
{json.dumps(claims_result, indent=2)}

Policy subagent result:
{json.dumps(policy_result, indent=2)}

Create one unified response containing:

1. Why the claim was rejected
2. Whether the service is covered
3. Any conflict between the claim decision and policy
4. Estimated insurer and member amounts, if supported
5. Missing or uncertain information
6. Recommended next action

Clearly separate confirmed facts from conclusions.
Do not invent information.
"""

    response = await asyncio.to_thread(
        client.messages.create,
        model=model,
        max_tokens=700,
        messages=[
            {
                "role": "user",
                "content": synthesis_prompt
            }
        ]
    )

    return "\n".join(
        block.text
        for block in response.content
        if block.type == "text"
    )


# ============================================================
# 6. RUN THE COORDINATOR
# ============================================================

answer = asyncio.run(
    coordinator(
        claim_id="CLM-1048",
        policy_id="GOLD-HOSPITAL",
        user_question=(
            "Why was CLM-1048 rejected, is the MRI covered, "
            "and what might the member need to pay?"
        )
    )
)

print(answer)